In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import muspan as ms
from itertools import zip_longest
from typing import List, Tuple, Optional
import math
from functools import partial
import seaborn as sns
from scipy import stats
import pathlib

In [ ]:
## Read in domains with Mets
# Make list of file names
directory = pathlib.Path("../outputs/In_out")
i=0
list_of_files = []
for item in directory.rglob("*"):
    if item.is_file():
            if str(item) != str(directory)+'/.DS_Store':
                list_of_files.append(str(item))
            i+=1
names = [x.replace('../domains_with_mets/', '').replace('.muspan', '') for x in list_of_files]

# make reference lists of met numbers and corresponding list of domain names (should really be a dict but it was too late to change it)
met_nos = [1,1,2,3,7,17,2,23,13,4]
domains_order = ['AKPT_Liver_Mets_4d_2_Region_1', 
                'AKPT_Liver_Mets_4d_3_Region_1', 
                'AKPT_Liver_Mets_4d_1_Region_2', 
                'AKPT_Liver_Mets_4d_5_Region_1',
                'AKPT_Liver_Mets_28d_2_Region_2', 
                'AKPT_Liver_Mets_28d_1_Region_1', 
                'AKPT_Liver_Mets_4d_1_Region_1',
                'AKPT_Liver_Mets_28d_2_Region_1',
                'AKPT_Liver_Mets_28d_3_Region_1',
                'AKPT_Liver_Mets_28d_1_Region_2']

# read in domains from file names
list_of_domains = []
for i in range(len(list_of_files)):
    list_of_domains.append(ms.io.load_domain(list_of_files[i], print_summary=False))

Calculate distance of every cell centre from each cell boundaries and save them for future reference. Distances saved separately for cells inside each Met, and those outside the Mets. For cells outside the Mets this saves both the distance to the nearest Met and also to each individual Met separately.

In [ ]:
## far more efficient method thanks to Padraig
for dom in range(len(list_of_domains)):
    domain = list_of_domains[dom]
    met_no = met_nos[domains_order.index(str(domain.name))]
    query_contained = ms.query.query(domain,('collection',), 'is', 'Cell Centres in Met 1')
    for region_no in range(met_no):
        shape_ID=ms.query.interpret_query(ms.query.query(domain, ('collection',),'is', 'Metastasis '+str(region_no+1)))
        shape_cells = ms.query.return_object_IDs_from_query_like(domain,('collection', 'Cell Centres in Met '+str(region_no+1)),bypass_checks=True)
        query_contained_temp = ms.query.query(domain,('collection',), 'is', 'Cell Centres in Met '+str(region_no+1))
        query_contained = ms.query.query_container(query_contained_temp, 'OR', query_contained, domain)
        distances = ms.helpers.object_to_object_distance_matrix(domain, population_A=shape_cells, population_B=shape_ID)
        shape_cell_distances = np.stack([shape_cells,distances.min(axis=1)])
    
        np.save('../outputs/Useful_Data/'+str(domain.name)+'_inside_Met_'+str(region_no+1), shape_cell_distances, allow_pickle=True)
    
    shapes_ID=ms.query.interpret_query(ms.query.query(domain, ('collection',),'is', 'Detailed Met Annotations'))
    query_centre = ms.query.query(domain, ('collection', ), 'is', 'Cell centres')
    query_not_contained = ms.query.query_container(query_centre, 'AND NOT', query_contained)
    outside_cells = ms.query.return_object_IDs_from_query_like(domain,query_not_contained,bypass_checks=True)
    for region_no in range(met_no):
        shape_ID = ms.query.query(domain, ('collection',),  'is', 'Metastasis '+str(region_no+1))
    
        distances = ms.helpers.object_to_object_distance_matrix(domain, population_A=outside_cells, population_B=shape_ID)
        outside_met_cell_distances = np.stack([outside_cells,distances.min(axis=1)])
    
        np.save('../outputs/Useful_Data/'+str(domain.name)+'_outside_to_Met_'+str(region_no+1), outside_met_cell_distances, allow_pickle=True)
    
    shapes = ms.query.query(domain, ('collection',),  'is', 'Detailed Met Annotations')
    distances = ms.helpers.object_to_object_distance_matrix(domain, population_A=outside_cells, population_B=shapes)
    outside_met_cell_distances = np.stack([outside_cells,distances.min(axis=1)])

    np.save('../outputs/Useful_Data/'+str(domain.name)+'_outside_to_all_Mets', outside_met_cell_distances, allow_pickle=True)
